# Chapter 02 — Never Stand in Front of the Steamroller

**Companion to *Applied AI*.**

This notebook accompanies Chapter 2. It is deliberately small. The chapter is
an argument supported by labour-market and productivity research, and a
notebook cannot add evidence to that argument.

What it *can* do is turn the chapter's closing exercise into something you can
run and edit.

## Question

**Which of my own tasks are exposed** — tasks whose value is the codified part,
where I hold none of the four jobs the chapter says cannot leave the human?

## What this notebook establishes

- A scoring rule for the chapter's exposure audit, applied to its own worked
  example, reproducing the chapter's "2 of 5 exposed".
- An editable block so you can score your own tasks.

## What this notebook does **not** establish

- **Nothing here predicts anyone's employment.** There is no model of the
  labour market, and building one from a five-row worksheet would be exactly
  the overreach the chapter warns against.
- The chapter's measured results — the 19% kept-pace shortfall, the 34% novice
  gain, the 19% jagged-frontier penalty — come from cited studies with stated
  bounds. They are **not** reproduced here, and this worksheet is not evidence
  for or against them.
- Your own scoring is a **self-report**, useful for thinking and worth nothing
  as data.

## The rule

The chapter's audit marks each task **C** (its value is the codified part) or
**T** (its value is tacit judgment), then asks which of the four jobs you
actually hold: **intent**, **authority**, **verification**, **frontier
judgment**.

A task is *exposed* when it is codified **and** you hold none of the four.

In [1]:
from dataclasses import dataclass, field
from typing import Set

FOUR_JOBS = ("intent", "authority", "verification", "frontier judgment")

@dataclass
class Task:
    name: str
    value: str                       # "C" codified, or "T" tacit
    jobs_held: Set[str] = field(default_factory=set)

    def __post_init__(self):
        assert self.value in ("C", "T"), "value must be 'C' or 'T'"
        unknown = self.jobs_held - set(FOUR_JOBS)
        assert not unknown, f"not one of the four jobs: {unknown}"

    @property
    def exposed(self) -> bool:
        return self.value == "C" and not self.jobs_held

def audit(tasks):
    rows = []
    for t in tasks:
        rows.append({
            "task": t.name,
            "C/T": t.value,
            "four-job held": ", ".join(sorted(t.jobs_held)) or "none",
            "exposed": "yes" if t.exposed else "no",
        })
    return rows

def show(rows):
    w = max([len(r["task"]) for r in rows] + [len("task")]) + 2
    j = max([len(r["four-job held"]) for r in rows] + [len("four-job held")]) + 2
    width = w + 6 + j + 7
    print(f"{'task':<{w}}{'C/T':<6}{'four-job held':<{j}}{'exposed'}")
    print("-" * width)
    for r in rows:
        print(f"{r['task']:<{w}}{r['C/T']:<6}{r['four-job held']:<{j}}{r['exposed']}")
    n = sum(1 for r in rows if r["exposed"] == "yes")
    print("-" * width)
    print(f"exposed: {n} of {len(rows)}")
    return n

## The chapter's worked example

These five tasks are the chapter's own illustrative sketch, not measured data.

In [2]:
chapter_example = [
    Task("Reset passwords per runbook", "C"),
    Task("Triage routine support tickets", "C"),
    Task("Write weekly status summary", "C", {"intent"}),
    Task("Judge an ambiguous outage escalation", "T", {"frontier judgment"}),
    Task("Sign off a production migration", "T", {"authority", "verification"}),
]

n_exposed = show(audit(chapter_example))
assert n_exposed == 2, "should reproduce the chapter's 2 of 5"

task                                  C/T   four-job held            exposed
----------------------------------------------------------------------------
Reset passwords per runbook           C     none                     yes
Triage routine support tickets        C     none                     yes
Write weekly status summary           C     intent                   no
Judge an ambiguous outage escalation  T     frontier judgment        no
Sign off a production migration       T     authority, verification  no
----------------------------------------------------------------------------
exposed: 2 of 5


## Observation

Two tasks are exposed, and notice *why* the third codified task is not. Writing
the weekly summary is just as codified as resetting passwords. It is not
exposed because someone holds **intent** over it: what the summary is for, and
what would make it a good one.

That is the chapter's point in one row. The protection is not the difficulty of
the task. It is whether one of the four jobs sits on top of it.

In [3]:
codified = [t for t in chapter_example if t.value == "C"]
print("codified tasks:", len(codified))
for t in codified:
    holds = ", ".join(sorted(t.jobs_held)) or "nothing"
    print(f"  {t.name:<38} held: {holds:<22} exposed: {t.exposed}")

codified tasks: 3
  Reset passwords per runbook            held: nothing                exposed: True
  Triage routine support tickets         held: nothing                exposed: True
  Write weekly status summary            held: intent                 exposed: False


## Your turn

Replace these with five things you were actually paid to do last month. Be
concrete: tasks, not a job title.

For each codified one, try to finish the chapter's sentence:

> *If a model did this at 90% quality for a hundredth of the cost, what would
> still need me?*

If you cannot finish it, that is the finding — and it is a finding about the
task, not about you.

In [4]:
my_tasks = [
    Task("(replace me) task 1", "C"),
    Task("(replace me) task 2", "C", {"intent"}),
    Task("(replace me) task 3", "T", {"verification"}),
    Task("(replace me) task 4", "C"),
    Task("(replace me) task 5", "T", {"authority", "frontier judgment"}),
]

mine = show(audit(my_tasks))
print()
print("Tasks to move off first (codified, no job held):")
for t in my_tasks:
    if t.exposed:
        print("  -", t.name)

task                 C/T   four-job held                 exposed
----------------------------------------------------------------
(replace me) task 1  C     none                          yes
(replace me) task 2  C     intent                        no
(replace me) task 3  T     verification                  no
(replace me) task 4  C     none                          yes
(replace me) task 5  T     authority, frontier judgment  no
----------------------------------------------------------------
exposed: 2 of 5

Tasks to move off first (codified, no job held):
  - (replace me) task 1
  - (replace me) task 4


## Interpretation

The count is not a score and it does not travel. It does one thing: it forces
the difference between *"I am busy"* and *"the value of this is the part that
is written down"*.

The chapter is careful that this advice has limits, and so should you be:

- **The ladder problem.** The route to senior judgment historically ran through
  the junior codified work. "Acquire senior judgment" is not actionable for
  someone who cannot get hired to acquire it.
- **The composition problem.** Advice that works for one person can be
  arithmetically impossible for a cohort.
- **It is a dated position.** The chapter files its own claim about the future
  under *predicted*, the weakest of its four evidence strengths.

## Try it yourself

1. **Score a colleague's role** rather than your own, then ask them. Where you
   disagree is usually where the tacit part actually lives.
2. **Re-score after adding a job.** Take one exposed task and ask what it would
   mean to hold *verification* over it — an actual check, not a feeling. Does
   the exposure flag clear, and did anything real change?
3. **Break the rule deliberately.** Change `exposed` to mark tasks with *any*
   job held as safe. Re-run. Does the result flatter you? The chapter's whole
   warning is that the flattering reading is the available one.